<a href="https://colab.research.google.com/github/mirrash7/Experiments/blob/main/mcbyte_tracker_comparison_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/roboflow-logomark-color.png)](https://roboflow.com)

# Test McByte on Your Own Video — and Compare It to Other Trackers

McByte is a **mask-conditioned multi-object tracker** now shipping in [`roboflow/trackers`](https://github.com/roboflow/trackers). It extends BoT-SORT-style association with temporally propagated **SAM + Cutie** segmentation masks as an extra matching cue, resolving ambiguous matches that IoU alone can't — with **no per-video tuning**.

This notebook lets you:

1. 📤 **Upload your own video** (or grab a sample)
2. 🎯 Run detection **once** with RF-DETR, then feed the *identical* detections to every tracker (a fair, apples-to-apples comparison)
3. 🏁 Compare **McByte** (both mask-free *and* full **SAM + Cutie** mask-conditioned) against **SORT, ByteTrack, OC-SORT, BoT-SORT, and C-BIoU**
4. 🎞️ Get per-tracker annotated videos, a **side-by-side comparison grid**, and a summary table (unique IDs, tracker-only FPS)
5. 🧪 Tune the mask pipeline's speed/quality trade-offs

**Reported benchmarks** (default params, HOTA — McByte vs. its BoT-SORT baseline, from [PR #513](https://github.com/roboflow/trackers/pull/513)):

| Dataset | BoT-SORT | McByte |
|---|---|---|
| DanceTrack | 57.8 | **67.2** |
| SportsMOT | 73.8 | **76.5** |
| SoccerNet | 84.5 | **85.0** |
| MOT17 | 63.7 | **64.1** |

Useful links: [Trackers docs](https://trackers.roboflow.com) · [McByte docs](https://trackers.roboflow.com/latest/trackers/mcbyte/) · [McByte paper (arXiv:2506.01373)](https://arxiv.org/abs/2506.01373) · [Original McByte repo](https://github.com/tstanczyk95/McByte)

> ⚡ **Runtime tip:** use a GPU runtime (`Runtime → Change runtime type → T4 GPU`). The lightweight comparison works on CPU too, but detection and the optional mask pipeline are much faster on GPU.

## 0. Check GPU

In [ ]:
!nvidia-smi || echo "No GPU detected — the notebook still works, just slower. The optional SAM+Cutie section really wants a GPU."


## 1. Install dependencies

We install:
- **`trackers`** — the tracking library (McByte requires `trackers>=2.6.0`)
- **`rfdetr`** — RF-DETR detector used to generate detections
- **`supervision`** — annotation + video utilities

In [ ]:
!pip install -q "trackers>=2.6.0" rfdetr supervision

from importlib.metadata import version
print("trackers:", version("trackers"))
print("supervision:", version("supervision"))
print("rfdetr:", version("rfdetr"))


## 2. Get a video

**Option A — upload your own** (run the cell below and pick a file), or
**Option B — use a sample video** (skip the upload dialog by pressing *Cancel*; the sample downloads automatically).

Short clips (5–20 s) keep the comparison quick. You can also cap the number of frames in the config cell.

In [ ]:
import os
from pathlib import Path

SOURCE_VIDEO_PATH = None

try:
    from google.colab import files  # noqa
    print("Upload a video file (mp4/mov/avi) — or press Cancel to use the sample video.")
    uploaded = files.upload()
    if uploaded:
        SOURCE_VIDEO_PATH = list(uploaded.keys())[0]
except Exception as e:
    print("Not running in Colab or upload skipped:", e)

if not SOURCE_VIDEO_PATH:
    from supervision.assets import download_assets, VideoAssets
    SOURCE_VIDEO_PATH = download_assets(VideoAssets.PEOPLE_WALKING)
    print("Using sample video.")

SOURCE_VIDEO_PATH = str(Path(SOURCE_VIDEO_PATH).resolve())
print("SOURCE_VIDEO_PATH =", SOURCE_VIDEO_PATH)


## 3. Configuration

- `MAX_FRAMES` — cap processed frames (`None` = whole video)
- `CONFIDENCE_THRESHOLD` — detections below this are discarded *before* tracking
- `CLASS_FILTER` — keep only these COCO class names (e.g. `["person"]`); `None` keeps everything
- `TRACKERS_TO_RUN` — which trackers to include in the comparison
- `RUN_MCBYTE_WITH_MASKS` — include **full-strength McByte** (SAM + Cutie mask-conditioned association) in the comparison. This is where McByte's benchmark gains come from, but it's *much* slower (~1–5 FPS tracking on a T4, and cost grows with the number of tracked objects). With it enabled, keep `MAX_FRAMES` modest (150–300).

In [ ]:
MAX_FRAMES = None           # e.g. 300 frames = 10 s @ 30 fps; set None for full video
CONFIDENCE_THRESHOLD = 0.3
CLASS_FILTER = ["person"]     # None to track all classes

RUN_MCBYTE_WITH_MASKS = True  # full SAM + Cutie pipeline — GPU strongly recommended

TRACKERS_TO_RUN = [
    "McByte + masks",   # mask-conditioned association (only runs if RUN_MCBYTE_WITH_MASKS)
    "McByte",           # mask-free: clear-match locking + IoU only
    "BoT-SORT",
    "ByteTrack",
    "OC-SORT",
    "SORT",
    "C-BIoU",
]


OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 4. Run detection once, reuse everywhere

To compare *trackers* — not detector randomness — we run **RF-DETR Medium once per frame** and cache the detections. Every tracker then consumes the exact same detections, so any difference you see in the output is purely down to the association algorithm.

In [ ]:
import cv2
import numpy as np
import supervision as sv
from tqdm.auto import tqdm
from rfdetr import RFDETRLarge

try:
    from rfdetr.util.coco_classes import COCO_CLASSES
except Exception:
    COCO_CLASSES = None

model = RFDETRLarge()

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
total = video_info.total_frames if MAX_FRAMES is None else min(MAX_FRAMES, video_info.total_frames)
print(video_info)

frames = []            # BGR frames (needed by BoT-SORT / McByte for camera motion compensation)
cached_detections = [] # one sv.Detections per frame

cap = cv2.VideoCapture(SOURCE_VIDEO_PATH)
for _ in tqdm(range(total), desc="Detecting (RF-DETR)"):
    ok, frame_bgr = cap.read()
    if not ok:
        break
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    detections = model.predict(frame_rgb, threshold=CONFIDENCE_THRESHOLD)

    if CLASS_FILTER and COCO_CLASSES is not None and detections.class_id is not None:
        keep = np.array(
            [COCO_CLASSES.get(int(c), str(c)) in CLASS_FILTER for c in detections.class_id]
        )
        detections = detections[keep]

    frames.append(frame_bgr)
    cached_detections.append(detections)
cap.release()

print(f"Cached {len(frames)} frames, "
      f"avg {np.mean([len(d) for d in cached_detections]):.1f} detections/frame")


## 5. Define the tracker line-up

Class names are resolved dynamically from the installed `trackers` version, so the notebook keeps working if a tracker isn't available in your build.

Two McByte configurations run side by side:

- **McByte** — lightweight, mask-free (clear-match locking + IoU association; behaves very close to BoT-SORT)
- **McByte + masks** — the full pipeline: **SAM** initializes a segmentation mask for each new track, **Cutie** propagates it frame-to-frame, and mask evidence resolves ambiguous IoU matches. This configuration is what produces the reported benchmark gains (e.g. DanceTrack 57.8 → 67.2 HOTA over BoT-SORT). SAM + Cutie checkpoints download automatically on first use.

BoT-SORT and both McByte variants receive the frame in `update()` so camera-motion compensation (and mask propagation) is active.

In [ ]:
import sys
import subprocess as sp

if RUN_MCBYTE_WITH_MASKS:
    print("Installing SAM + Cutie mask dependencies (trackers[mask]) ...")
    sp.run([sys.executable, "-m", "pip", "install", "-q", "trackers[mask]"], check=True)
    print("Done.")


In [ ]:
import inspect
import trackers

CANDIDATE_CLASSES = {
    "SORT":           ["SORTTracker", "SortTracker"],
    "ByteTrack":      ["ByteTrackTracker", "ByteTracker"],
    "OC-SORT":        ["OCSORTTracker", "OcSortTracker"],
    "BoT-SORT":       ["BoTSORTTracker", "BotSortTracker"],
    "C-BIoU":         ["CBIoUTracker", "CBIOUTracker", "CBiouTracker"],
    "McByte":         ["McByteTracker"],
    "McByte + masks": ["McByteTracker"],
}

def draw_tracker_label(img, text, org=(16, 16), font_scale=1.4, thickness=3,
                       bg_color=(128, 0, 128),      # purple (BGR)
                       text_color=(255, 255, 255),  # white
                       pad=14, radius=18, alpha=0.85):
    """Draw text on a rounded-corner filled badge. Modifies img in place."""
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), baseline = cv2.getTextSize(text, font, font_scale, thickness)
    x, y = org
    bw, bh = tw + 2 * pad, th + baseline + 2 * pad
    overlay = img.copy()
    # rounded rectangle = two overlapping rects + four corner circles
    cv2.rectangle(overlay, (x + radius, y), (x + bw - radius, y + bh), bg_color, -1)
    cv2.rectangle(overlay, (x, y + radius), (x + bw, y + bh - radius), bg_color, -1)
    for cx, cy in [(x + radius, y + radius), (x + bw - radius, y + radius),
                   (x + radius, y + bh - radius), (x + bw - radius, y + bh - radius)]:
        cv2.circle(overlay, (cx, cy), radius, bg_color, -1)
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)
    cv2.putText(img, text, (x + pad, y + pad + th), font, font_scale,
                text_color, thickness, cv2.LINE_AA)
    return img

def resolve_tracker_class(name):
    for cls_name in CANDIDATE_CLASSES[name]:
        cls = getattr(trackers, cls_name, None)
        if cls is not None:
            return cls
    return None

def make_tracker(name):
    cls = resolve_tracker_class(name)
    if cls is None:
        return None
    kwargs = {}
    sig = inspect.signature(cls.__init__)
    if "frame_rate" in sig.parameters:
        kwargs["frame_rate"] = float(video_info.fps)
    if name == "McByte + masks":
        from trackers import McByteMaskConfig
        kwargs["enable_mask_manager"] = True
        kwargs["mask_config"] = McByteMaskConfig(device="auto")  # SAM + Cutie on GPU if available
        # kwargs["enable_isolated_mask_matching"] = True  # optional: recover more matches under heavy occlusion
    return cls(**kwargs)

def tracker_enabled(name):
    if name == "McByte + masks" and not RUN_MCBYTE_WITH_MASKS:
        return False
    return resolve_tracker_class(name) is not None

available = [n for n in TRACKERS_TO_RUN if tracker_enabled(n)]
missing = [n for n in TRACKERS_TO_RUN if n not in available]
print("Will run:", available)
if missing:
    print("Skipped (disabled or not found in this trackers version):", missing)


## 6. Run all trackers

For each tracker we replay the cached detections, time **only the tracker's `update()` call**, and write an annotated video (boxes + persistent IDs + motion traces, colored per track ID).

> ⏳ **"McByte + masks" is the slow one.** Its first frames also include SAM/Cutie checkpoint downloads and model warm-up, so its reported FPS is a slight underestimate. The mask-free trackers each finish in seconds.

In [ ]:
import copy
import time
import subprocess

def safe_name(name):
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in name)

def run_tracker(name):
    tracker = make_tracker(name)
    accepts_frame = "frame" in inspect.signature(tracker.update).parameters

    box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
    label_annotator = sv.LabelAnnotator(
        color_lookup=sv.ColorLookup.TRACK, text_scale=0.5
    )
    trace_annotator = sv.TraceAnnotator(
        color_lookup=sv.ColorLookup.TRACK, trace_length=60
    )

    raw_path = os.path.join(OUTPUT_DIR, f"{safe_name(name)}_raw.mp4")
    out_path = os.path.join(OUTPUT_DIR, f"{safe_name(name)}.mp4")
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(
        raw_path, cv2.VideoWriter_fourcc(*"mp4v"), video_info.fps, (w, h)
    )

    unique_ids = set()
    tracked_per_frame = []
    update_time = 0.0
    annotated_frames = []

    for frame_bgr, detections in tqdm(
        list(zip(frames, cached_detections)), desc=name, leave=False
    ):
        detections = copy.deepcopy(detections)  # trackers may mutate detections

        t0 = time.perf_counter()
        if accepts_frame:
            tracked = tracker.update(detections, frame=frame_bgr)
        else:
            tracked = tracker.update(detections)
        update_time += time.perf_counter() - t0

        annotated = frame_bgr.copy()
        if tracked.tracker_id is not None and len(tracked) > 0:
            ids = [int(t) for t in tracked.tracker_id]
            unique_ids.update(ids)
            tracked_per_frame.append(len(tracked))
            labels = [f"#{t}" for t in ids]
            annotated = trace_annotator.annotate(annotated, tracked)
            annotated = box_annotator.annotate(annotated, tracked)
            annotated = label_annotator.annotate(annotated, tracked, labels=labels)
        else:
            tracked_per_frame.append(0)

        draw_tracker_label(annotated, name)
        writer.write(annotated)
        annotated_frames.append(annotated)

    writer.release()
    # re-encode to H.264 so the video plays inline in the browser
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", raw_path,
         "-vcodec", "libx264", "-pix_fmt", "yuv420p", out_path],
        check=True,
    )
    os.remove(raw_path)

    return {
        "name": name,
        "video": out_path,
        "frames": annotated_frames,
        "unique_ids": len(unique_ids),
        "avg_tracked_per_frame": float(np.mean(tracked_per_frame)) if tracked_per_frame else 0.0,
        "tracker_fps": len(frames) / update_time if update_time > 0 else float("inf"),
    }

results = {}
for name in available:
    results[name] = run_tracker(name)
    r = results[name]
    print(f"{name:>10}: {r['unique_ids']} unique IDs | "
          f"{r['avg_tracked_per_frame']:.1f} tracks/frame | "
          f"tracker-only {r['tracker_fps']:.0f} FPS")


## 7. Results

### Summary table

A few caveats on reading these numbers **without ground truth**:

- **Unique IDs** — with a fixed number of real objects, *fewer* unique IDs usually means fewer identity switches / fragmented tracks. (Too few could also mean merged identities — check the videos.)
- **Avg tracks/frame** — how much of the scene the tracker keeps locked on each frame.
- **Tracker-only FPS** — speed of the association step alone (detection excluded).

**What to expect:**
- Mask-free **McByte ≈ BoT-SORT** — that's by design; without masks, McByte is a clear-match-locking variant of the same association pipeline.
- **McByte + masks** is where the differences should appear: watch moments where objects **cross, occlude, or look alike**. Mask evidence should keep IDs stable through exactly those events. On easy footage with well-separated objects, all good trackers will look similar — masks earn their keep in crowded, DanceTrack-style scenes.
- Its FPS will be 1–2 orders of magnitude lower — that's the cost of running Cutie mask propagation every frame.

For proper **HOTA / MOTA / IDF1** metrics you need ground-truth annotations — see the [evaluation guide](https://trackers.roboflow.com/latest/evaluations/evaluate/) and `trackers eval`.

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {
        "Tracker": r["name"],
        "Unique track IDs": r["unique_ids"],
        "Avg tracks/frame": round(r["avg_tracked_per_frame"], 1),
        "Tracker-only FPS": round(r["tracker_fps"], 0),
    }
    for r in results.values()
]).set_index("Tracker")

df


### Watch individual tracker outputs

Pick any tracker and play its annotated video inline. Follow a few objects through occlusions and watch whether their `#ID` labels survive.

In [ ]:
import base64
from IPython.display import HTML, display

def show_video(path, width=720):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    return HTML(
        f'<video width="{width}" controls muted loop>'
        f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'
    )

TRACKER_TO_SHOW = "McByte + masks" if "McByte + masks" in results else "McByte"
# change to any tracker name from the table above

display(show_video(results[TRACKER_TO_SHOW]["video"]))


### Visualize McByte's SAM+Cutie propagated masks (ID-matched colors)
Each mask is matched to the tracked box it overlaps most and painted with that track's ID color — the same palette.by_idx(tracker_id) that sv.ColorLookup.TRACK uses for boxes/labels — so mask, box, and #ID label all share one color per identity. Unmatched masks are drawn gray.

Masks only appear after a track has been visible for minimum_mask_creation_frames (default 3), so early frames show zero masks.

In [ ]:
MASK_VIS_FRAMES = len(frames)   # or cap it, e.g. min(150, len(frames))

def _to_numpy(obj):
    try:
        import torch as _t
        if isinstance(obj, _t.Tensor):
            obj = obj.detach().cpu().numpy()
    except Exception:
        pass
    return obj if isinstance(obj, np.ndarray) else None

def _split_label_map(m):
    """(H, W) integer label map -> (N, H, W) binary masks, one per object id > 0."""
    vals = np.unique(m)
    vals = vals[vals > 0]
    if len(vals) == 0:
        return None
    return np.stack([(m == v).astype(np.float32) for v in vals])

def _normalize_masks(arr):
    """Any mask-like array -> (N, H, W) float32 per-object binary masks, or None."""
    arr = _to_numpy(arr)
    if arr is None:
        return None
    arr = np.squeeze(arr)
    if arr.ndim == 2:
        u = np.unique(arr)
        if len(u) > 2 and np.allclose(u, np.round(u)):
            return _split_label_map(arr)              # integer label map
        return (arr > 0.5).astype(np.float32)[None]   # single binary/prob mask
    if arr.ndim == 3 and min(arr.shape[1:]) > 4:
        # stack of masks — but each layer might itself be a label map
        if arr.shape[0] == 1:
            return _normalize_masks(arr[0])
        return (arr > 0.5).astype(np.float32)
    return None

def extract_masks(tracker, frame_shape, verbose=False):
    manager = getattr(tracker, "mask_manager", None) or getattr(tracker, "_mask_manager", None)
    if manager is None:
        if verbose:
            print("No mask manager — was the tracker built with enable_mask_manager=True?")
        return None

    holders = [manager]
    for attr in ("propagator", "_propagator", "mask_propagator", "_mask_propagator"):
        holders.append(getattr(manager, attr, None))

    candidates = []
    for holder in filter(None, holders):
        for attr in dir(holder):
            if attr.startswith("__") or "mask" not in attr.lower():
                continue
            try:
                val = getattr(holder, attr)
            except Exception:
                continue
            if callable(val):
                continue
            for source in (val, getattr(val, "masks", None)):
                m = _normalize_masks(source)
                if m is not None:
                    candidates.append((f"{type(holder).__name__}.{attr}", m))
            if isinstance(val, dict) and val:
                ms = [_normalize_masks(v) for v in val.values()]
                ms = [m[0] for m in ms if m is not None and m.shape[0] == 1]
                if ms and len({m.shape for m in ms}) == 1:
                    candidates.append((f"{type(holder).__name__}.{attr}[dict]", np.stack(ms)))

    if verbose:
        print("mask sources found:",
              [f"{c[0]} -> {c[1].shape[0]} object masks" for c in candidates] or "none yet")
    if not candidates:
        return None

    H, W = frame_shape[:2]
    full = [c for c in candidates if c[1].shape[1:] == (H, W)]
    name, masks = max(full or candidates, key=lambda c: c[1].shape[0])
    if masks.shape[1:] != (H, W):
        masks = np.stack([cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
                          for m in masks])
    return masks

def match_masks_to_tracks(masks, tracked, min_frac=0.30):
    """Greedy one-to-one: each mask -> tracker_id of the box holding most of it."""
    ids = [None] * len(masks)
    if tracked.tracker_id is None or len(tracked) == 0:
        return ids
    boxes = tracked.xyxy.astype(int)
    pairs = []  # (containment, mask_idx, box_idx)
    for k, m in enumerate(masks):
        binary = m > 0.5
        area = binary.sum()
        if area == 0:
            continue
        for b, (x1, y1, x2, y2) in enumerate(boxes):
            x1, y1 = max(x1, 0), max(y1, 0)
            if x2 <= x1 or y2 <= y1:
                continue
            frac = binary[y1:y2, x1:x2].sum() / area
            if frac >= min_frac:
                pairs.append((frac, k, b))
    used_masks, used_boxes = set(), set()
    for frac, k, b in sorted(pairs, reverse=True):   # best matches claim first
        if k in used_masks or b in used_boxes:
            continue
        ids[k] = int(tracked.tracker_id[b])
        used_masks.add(k); used_boxes.add(b)
    return ids

viz_tracker = make_tracker("McByte + masks")
assert viz_tracker is not None, "Set RUN_MCBYTE_WITH_MASKS = True and re-run Section 5 first."

palette = sv.ColorPalette.DEFAULT          # same palette ColorLookup.TRACK uses
UNMATCHED_COLOR = (128, 128, 128)
box_annotator = sv.BoxAnnotator(color_lookup=sv.ColorLookup.TRACK)
label_annotator = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK, text_scale=0.5)

raw_path = os.path.join(OUTPUT_DIR, "mcbyte_mask_viz_raw.mp4")
out_path = os.path.join(OUTPUT_DIR, "mcbyte_mask_viz.mp4")
h, w = frames[0].shape[:2]
writer = cv2.VideoWriter(raw_path, cv2.VideoWriter_fourcc(*"mp4v"), video_info.fps, (w, h))

for i, (frame_bgr, detections) in enumerate(tqdm(
    list(zip(frames[:MASK_VIS_FRAMES], cached_detections[:MASK_VIS_FRAMES])),
    desc="McByte mask viz",
)):
    detections = copy.deepcopy(detections)
    tracked = viz_tracker.update(detections, frame=frame_bgr)

    masks = extract_masks(viz_tracker, frame_bgr.shape, verbose=(i == 10))

    annotated = frame_bgr.copy()
    n_masks = 0
    if masks is not None:
        mask_track_ids = match_masks_to_tracks(masks, tracked)
        overlay = annotated.copy()
        for m, tid in zip(masks, mask_track_ids):
            binary = (m > 0.5).astype(np.uint8)
            if binary.sum() == 0:
                continue
            n_masks += 1
            color = palette.by_idx(tid).as_bgr() if tid is not None else UNMATCHED_COLOR
            overlay[binary.astype(bool)] = color
            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                           cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(annotated, contours, -1, color, 2)
        annotated = cv2.addWeighted(overlay, 0.35, annotated, 0.65, 0)

    if tracked.tracker_id is not None and len(tracked) > 0:
        labels = [f"#{int(t)}" for t in tracked.tracker_id]
        annotated = box_annotator.annotate(annotated, tracked)
        annotated = label_annotator.annotate(annotated, tracked, labels=labels)

    text = f"McByte"
    draw_tracker_label(annotated, text)
    writer.write(annotated)

writer.release()
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw_path,
                "-vcodec", "libx264", "-pix_fmt", "yuv420p", out_path], check=True)
os.remove(raw_path)

import base64
from IPython.display import HTML, display

def show_video(path, width=720):
    with open(path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    return HTML(f'<video width="{width}" controls muted loop>'
                f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

display(show_video(out_path))

### Side-by-side comparison grid

All trackers tiled into one video so you can spot ID switches and lost tracks at a glance.

In [ ]:
import math

names = list(results.keys())
cols = 3 if len(names) > 2 else len(names)
rows = math.ceil(len(names) / cols)

h, w = frames[0].shape[:2]
tile_w = 640
tile_h = int(h * tile_w / w)
# keep even dimensions for the H.264 encoder
tile_w -= tile_w % 2
tile_h -= tile_h % 2

grid_raw = os.path.join(OUTPUT_DIR, "comparison_raw.mp4")
grid_path = os.path.join(OUTPUT_DIR, "comparison.mp4")
writer = cv2.VideoWriter(
    grid_raw, cv2.VideoWriter_fourcc(*"mp4v"),
    video_info.fps, (tile_w * cols, tile_h * rows),
)

n_frames = min(len(results[n]["frames"]) for n in names)
for i in tqdm(range(n_frames), desc="Building grid"):
    grid = np.zeros((tile_h * rows, tile_w * cols, 3), dtype=np.uint8)
    for k, name in enumerate(names):
        r, c = divmod(k, cols)
        tile = cv2.resize(results[name]["frames"][i], (tile_w, tile_h))
        grid[r * tile_h:(r + 1) * tile_h, c * tile_w:(c + 1) * tile_w] = tile
    writer.write(grid)
writer.release()

subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-i", grid_raw,
     "-vcodec", "libx264", "-pix_fmt", "yuv420p", grid_path],
    check=True,
)
os.remove(grid_raw)

display(show_video(grid_path, width=960))


### Download the results

In [ ]:
try:
    from google.colab import files
    files.download(os.path.join(OUTPUT_DIR, "comparison.mp4"))
    # uncomment to also download individual tracker videos:
    # for r in results.values():
    #     files.download(r["video"])
except Exception as e:
    print("Not in Colab — videos are in the ./outputs folder.", e)


## 8. Tuning the mask pipeline 🧪

"McByte + masks" ran with defaults above. Knobs worth trying if you want to trade quality vs. speed, then re-run Section 6:

```python
# inside make_tracker(), for "McByte + masks":
kwargs["mask_config"] = McByteMaskConfig(
    device="auto",
    cutie_max_internal_size=360,   # default 480 — lower = faster, coarser masks
    sam_model_type="vit_b",        # default; smallest SAM variant
    cutie_mem_every=15,            # default 10 — higher = faster memory updates
)
kwargs["enable_isolated_mask_matching"] = True  # rescue low-IoU matches under heavy
                                                # occlusion (may add false positives)
```

- **Mask evidence only fires on ambiguity.** Clear one-to-one matches are locked before masks are consulted, so on easy footage the mask run can legitimately look identical to mask-free. Crowded scenes with crossings are where you'll see it act.
- If the mask install fails, the manual SAM/Cutie setup steps are in [PR #513](https://github.com/roboflow/trackers/pull/513).

## 9. Going further

- **Rigorous metrics on benchmark data:** `trackers download --name mot17 --split val --asset annotations,detections` then `trackers eval` for HOTA/MOTA/IDF1 — [evaluation guide](https://trackers.roboflow.com/latest/evaluations/evaluate/)
- **Tune hyperparameters** on your footage with Optuna — [tuning guide](https://trackers.roboflow.com/latest/guides/tune/)
- **Speed up the mask pipeline:** lower `McByteMaskConfig(cutie_max_internal_size=...)`, pick a smaller SAM variant (`sam_model_type`), or raise `cutie_mem_every` — [performance notes](https://trackers.roboflow.com/latest/trackers/mcbyte/)
- **CLI one-liner:** `trackers track --source video.mp4 --tracker mcbyte --output.video output.mp4`
- **Swap the detector:** anything that outputs `supervision.Detections` works — YOLO, DETR, your custom Roboflow model, etc.

Questions? [Trackers Discord](https://discord.gg/GbfgXGJ8Bk) · [GitHub issues](https://github.com/roboflow/trackers/issues)